In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
catalog = "ete"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"
data_source = "gross_price"

df_bronze = spark.table(f"{catalog}.{bronze_schema}.{data_source}")

print("Raw Bronze Pricing Data:")
display(df_bronze.limit(10))

In [0]:
df_silver = (
    df_bronze
    .withColumn(
        "month",
        F.coalesce(
            F.try_to_date(F.col("month"), "yyyy/MM/dd"),
            F.try_to_date(F.col("month"), "dd/MM/yyyy"),
            F.try_to_date(F.col("month"), "yyyy-MM-dd"),
            F.try_to_date(F.col("month"), "dd-MM-yyyy")
        )
    )
    .withColumn(
        "gross_price",
        F.when(
            F.col("gross_price").cast("string").rlike(r'^-?\d+(\.\d+)?$'), 
            F.when(F.col("gross_price").cast("double") < 0, -1 * F.col("gross_price").cast("double"))
             .otherwise(F.col("gross_price").cast("double"))
        ).otherwise(0) 
    )
    .withColumnRenamed("product_id", "product_code")
)

print("Cleaned Silver Pricing Data:")
display(df_silver)


silver_table_name = f"{catalog}.{silver_schema}.{data_source}"

df_silver.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(silver_table_name)

print(f" Silver pricing successfully written to {silver_table_name}")

### Gold Processing...

In [0]:
df_gold_price = (
    df_silver
    .withColumn("year", F.year("month"))
    .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))
)
window_spec = (
    Window
    .partitionBy("product_code", "year")
    .orderBy(F.col("is_zero"), F.col("month").desc())
)

df_gold_final = (
    df_gold_price
    .withColumn("rnk", F.row_number().over(window_spec))
    .filter(F.col("rnk") == 1)
)

In [0]:
df_gold_final = (
    df_gold_final
    .select("product_code", "year", "gross_price")
    .withColumnRenamed("gross_price", "price_usd")
    .withColumn("year", F.col("year").cast("string"))
)

print("Child Gold Pricing Data:")
display(df_gold_final)

child_gold_table = f"{catalog}.{gold_schema}.sb_dim_{data_source}"

df_gold_final.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(child_gold_table)

print(f" Child Gold saved to {child_gold_table}")

In [0]:
parent_table_name = f"{catalog}.{gold_schema}.dim_gross_price"
delta_table = DeltaTable.forName(spark, parent_table_name)
df_child_prices = spark.table(child_gold_table)

delta_table.alias("target").merge(
    source=df_child_prices.alias("source"),
    condition="target.product_code = source.product_code AND target.year = source.year"
).whenMatchedUpdate(
    set={
        "price_usd": "source.price_usd"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_usd": "source.price_usd",
        "year": "source.year"
    }
).execute()

print(" Pricing perfectly merged into parent table!")